# TurboVLA ALOHA -- Colab A100 full finetune + eval + latency (TIP-007)

Runs the **same local code** as TIP-002..006 (`experiments/aloha/*.py`,
`turbovla/data/lerobot_aloha.py`) on a Colab A100, just with a different
config: all 130 episodes, DINOv3 unfrozen, bigger batch, wandb online, and a
real train/val split (generalization, not just the overfit gate).

**This notebook does not reimplement any training/eval/latency logic** -- it
only clones the repo, installs deps, downloads assets, and calls the existing
`experiments.aloha.finetune` / `eval_openloop` / `profile_latency` /
`compute_stats` modules as subprocesses with Colab-appropriate arguments.

**No Google Drive mount.** Results are pulled out at the end via
`google.colab.files.download` (see the last section) -- **download them
immediately after the run finishes**. If the Colab runtime disconnects or
idle-times-out before you download, everything in `/content` is lost.

Prerequisites:
- Colab runtime set to **A100** (Runtime > Change runtime type > A100 GPU).
- A Hugging Face account with access to `facebook/dinov3-vitb16-pretrain-lvd1689m`
  approved (gated repo -- see TIP-001; if not yet approved, request access on
  the model page and wait before running the download cell).


## 1. GPU check

In [ ]:
!nvidia-smi


## 1 (cont). Clone repo + install

`pip install -e .` with **no extras** (no `.[libero]`/`.[robotwin]`), then
force torch to the cu121 CUDA build (Colab's default torch may already be
CUDA-enabled, but we pin to cu121 to match the rest of this project's TIPs),
then the same small extra deps used locally in TIP-001 -- **no TensorFlow,
no deepspeed, no flash-attn**.


In [ ]:
import os

REPO_URL = "https://github.com/DuyBaoDOCer/TurboVLA.git"  # Homeowner's fork/origin
REPO_BRANCH = "main"
REPO_DIR = "/content/TurboVLA"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}


In [ ]:
# Confirm exactly which branch/commit this runtime is using -- Colab only ever
# sees pushed code, never local files from the Windows dev machine, so this
# is the ground truth for "what am I actually running."
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD
!git log -1 --oneline


In [ ]:
!pip install -e . -q
!pip install --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install "av" "huggingface_hub" "pandas" "pyarrow" "matplotlib" "wandb" -q


In [ ]:
# Sanity: CUDA build correct, av importable, TensorFlow NOT required for the
# modules this notebook actually calls (compute_stats / lerobot_aloha /
# finetune / eval_openloop / profile_latency all avoid the TF-importing
# turbovla.data.libero_rlds / turbovla.training.trainer modules -- see
# TIP-002/003 Completion Reports for why).
import torch
print("torch:", torch.__version__, "cuda:", torch.version.cuda, "available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "CUDA not available -- check Runtime > Change runtime type > A100 GPU"

import av
print("av:", av.__version__)

try:
    from experiments.aloha.compute_stats import parse_episode_list
    from turbovla.data.lerobot_aloha import AlohaLeRobotDataset, split_by_episode
    from turbovla.models.turbovla import build_turbovla
    print("core imports OK (no TensorFlow required)")
except ModuleNotFoundError as exc:
    raise RuntimeError(f"unexpected import failure: {exc}") from exc


## 2. Auth -- Hugging Face token + wandb key (via Colab Secrets, never hardcoded)

This notebook lives in a **public/forked git repo**, so no token or API key is
ever written into it directly -- both come from **Colab Secrets** at runtime.

**Before running the cell below:** open the key icon (Secrets) in the left
sidebar of Colab and add two secrets:
- `HF_TOKEN` -- a Hugging Face access token with approved access to the gated
  `facebook/dinov3-vitb16-pretrain-lvd1689m` repo (see TIP-001). Required --
  asset downloads below will fail without it.
- `WANDB_API_KEY` -- your key from [wandb.ai/authorize](https://wandb.ai/authorize).
  Optional -- if you skip this, training still runs with `WANDB_MODE=disabled`
  and loss is still recorded to CSV/JSONL (TIP-003), just no wandb dashboard.

For each secret, toggle **"Notebook access"** on after adding it, or
`userdata.get(...)` will raise a permission error even though the secret
exists.


In [ ]:
import os
from google.colab import userdata

# HF_TOKEN: required. hf CLI / huggingface_hub both read this env var
# automatically for authenticated downloads (gated DINOv3 repo, TIP-001).
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print("HF_TOKEN set and logged in.")

# WANDB_API_KEY: optional. Falls back to WANDB_MODE=disabled (never blocks
# training) if the secret isn't configured -- CSV/JSONL loss logs (TIP-003)
# are unaffected either way.
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_MODE"] = "online"
    print("WANDB_API_KEY set -> WANDB_MODE=online.")
except Exception:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY chua set trong Colab Secrets -> chay WANDB_MODE=disabled "
          "(loss van duoc ghi CSV/JSONL, TIP-003).")


## 2 (cont). Download assets on the Colab network

DINOv3 + BERT backbones (HF cache), the **full 130-episode** ALOHA dataset,
and the `object.pth` init checkpoint. Colab's network is normally solid
enough for a one-shot download, but each download is wrapped in the same
resumable retry loop learned the hard way in TIP-001 (local network kept
dropping large transfers mid-stream) in case Colab's network has a bad
moment too.


In [ ]:
%%bash
retry_hf_download () {
  # $@ = full "hf download ..." argument list
  for attempt in $(seq 1 10); do
    hf download "$@" && return 0
    echo "attempt $attempt failed, retrying (resumes from where it left off)..."
  done
  echo "FAILED after 10 attempts: hf download $@" >&2
  return 1
}

retry_hf_download facebook/dinov3-vitb16-pretrain-lvd1689m
retry_hf_download google-bert/bert-base-uncased
retry_hf_download thanh0210/aloha_left_arm_pick_carrot_put_cup_easy_task --repo-type dataset --local-dir data/aloha
retry_hf_download H-EmbodVis/TurboVLA "checkpoints/libero/object.pth" --local-dir pretrained/TurboVLA


In [ ]:
# Report sizes actually downloaded.
!du -sh data/aloha pretrained/TurboVLA 2>/dev/null
!du -sh ~/.cache/huggingface/hub/models--facebook--dinov3-vitb16-pretrain-lvd1689m 2>/dev/null
!du -sh ~/.cache/huggingface/hub/models--google-bert--bert-base-uncased 2>/dev/null
!find data/aloha/data -name "*.parquet" | wc -l


## 3. Train/val split + stats computed on the TRAIN split only

`split_by_episode` (TIP-002) splits by whole episode (never by frame) so no
trajectory leaks across train/val. Stats (`aloha_stats_full.json`,
`stats_key="aloha_full"`) are computed with `compute_stats.py` on
**`train_eps` only** -- val must never influence normalization.


In [ ]:
from turbovla.data.lerobot_aloha import split_by_episode

ALL_EPISODES = list(range(130))
VAL_RATIO = 0.15
SPLIT_SEED = 42

train_eps, val_eps = split_by_episode(ALL_EPISODES, val_ratio=VAL_RATIO, seed=SPLIT_SEED)
print(f"train episodes: {len(train_eps)}  val episodes: {len(val_eps)}")
assert set(train_eps).isdisjoint(val_eps)
assert set(train_eps) | set(val_eps) == set(ALL_EPISODES)

train_eps_str = ",".join(map(str, train_eps))
val_eps_str = ",".join(map(str, val_eps))
STATS_PATH = "experiments/aloha/configs/aloha_stats_full.json"
STATS_KEY = "aloha_full"


In [ ]:
import subprocess

def run(cmd, **kwargs):
    """Runs a subprocess with real-time output; raises (halting notebook
    execution under Run All) on nonzero exit -- this is the actual
    enforcement mechanism for gates like the smoke-must-pass-before-full
    requirement, not just a printed warning."""
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)

run([
    "python", "-m", "experiments.aloha.compute_stats",
    "--data_root", "data/aloha",
    "--episodes", train_eps_str,
    "--out", STATS_PATH,
    "--stats_key", STATS_KEY,
])


## 4. Smoke gate (runs ON the A100 runtime -- do NOT switch to T4)

Quick sanity pass on 8 episodes, both backbones frozen, ~300 steps, wandb
disabled. This is the same `experiments.aloha.finetune` module TIP-003
validated locally -- here it's just confirming paths/deps/init/loss-direction
are correct **on this Colab environment** before spending A100 compute on the
full run. `finetune.py` itself already raises `RuntimeError` if the init
missing/unexpected keys deviate from the expected 3-key
`state_projection.net.{0.weight,0.bias,1.weight}` mismatch (TIP-003 logic,
unchanged) -- combined with `run()`'s `check=True` above, that failure would
stop this notebook here rather than silently continuing to the full run.


In [ ]:
run([
    "python", "-m", "experiments.aloha.finetune",
    "--data_root", "data/aloha",
    "--episodes", "0-7",
    "--stats_path", STATS_PATH,
    "--stats_key", STATS_KEY,
    "--init_ckpt", "pretrained/TurboVLA/checkpoints/libero/object.pth",
    "--max_steps", "300",
    "--batch_size", "4",
    "--grad_accum_steps", "1",
    "--warmup_steps", "30",
    "--wandb_mode", "disabled",
    "--run_dir", "experiments/aloha/runs/colab_smoke",
])
print("\nSMOKE GATE PASSED (init keys matched the expected 3-key deviation, no crash, ran to completion).")


## 5. Full A100 finetune (D10: unfreeze DINOv3, freeze BERT)

Edit `FULL_RUN_HPARAMS` below to tune batch size / steps / learning rates for
your A100 allocation before running. Defaults: batch_size=16 (no grad accum
needed on 40GB), ~5000 steps, `dinov3_lr=5e-5` (small, since DINOv3 is now
trainable), `head_lr=2e-4` (same as local smoke), BERT stays frozen.
`--stop_loss 0` disables the 5e-3 early-stop gate used for the local smoke
run (that threshold doesn't apply here -- generalization is judged by val
NMSE in the eval section below, not by training loss). `--wandb_mode` reads
whatever the auth cell resolved (`online` if `WANDB_API_KEY` was set in Colab
Secrets, `disabled` otherwise) -- never hardcoded, so this cell doesn't hang
or fail if the secret wasn't configured.


In [ ]:
FULL_RUN_HPARAMS = {
    "batch_size": "16",
    "grad_accum_steps": "1",
    "max_steps": "5000",
    "warmup_steps": "200",
    "save_steps": "1000",
    "head_lr": "2e-4",
    "dinov3_lr": "5e-5",
}

FULL_RUN_DIR = "experiments/aloha/runs/colab_full"
full_run_wandb_mode = os.environ.get("WANDB_MODE", "disabled")
print(f"wandb mode for full run: {full_run_wandb_mode}")

run([
    "python", "-m", "experiments.aloha.finetune",
    "--data_root", "data/aloha",
    "--episodes", train_eps_str,
    "--stats_path", STATS_PATH,
    "--stats_key", STATS_KEY,
    "--init_ckpt", "pretrained/TurboVLA/checkpoints/libero/object.pth",
    "--no_freeze_vision_encoder",   # D10: unfreeze DINOv3
    # --no_freeze_text_encoder is intentionally NOT passed: BERT stays frozen (D10)
    "--batch_size", FULL_RUN_HPARAMS["batch_size"],
    "--grad_accum_steps", FULL_RUN_HPARAMS["grad_accum_steps"],
    "--max_steps", FULL_RUN_HPARAMS["max_steps"],
    "--warmup_steps", FULL_RUN_HPARAMS["warmup_steps"],
    "--save_steps", FULL_RUN_HPARAMS["save_steps"],
    "--head_lr", FULL_RUN_HPARAMS["head_lr"],
    "--dinov3_lr", FULL_RUN_HPARAMS["dinov3_lr"],
    "--stop_loss", "0",             # disable the smoke-style early stop
    "--wandb_mode", full_run_wandb_mode,
    "--wandb_project", "turbovla-aloha-full",
    "--run_dir", FULL_RUN_DIR,
])


## Locate the final checkpoint

`finetune.py` saves `{checkpoint_prefix}_{global_step}.pth` under
`<run_dir>/checkpoints/` (no separate "final" file is produced by the
existing script, and this notebook does not add one -- per TIP-007
CONSTRAINTS it must not rewrite finetune.py's logic). This just finds the
highest-step checkpoint written by the run above.


In [ ]:
import glob, os, re

def find_latest_checkpoint(run_dir):
    candidates = glob.glob(os.path.join(run_dir, "checkpoints", "*_*.pth"))
    if not candidates:
        raise FileNotFoundError(f"no checkpoints found under {run_dir}/checkpoints")
    def step_of(path):
        m = re.search(r"_(\d+)\.pth$", os.path.basename(path))
        return int(m.group(1)) if m else -1
    return max(candidates, key=step_of)

final_ckpt = find_latest_checkpoint(FULL_RUN_DIR)
print("final checkpoint:", final_ckpt)
print(f"size: {os.path.getsize(final_ckpt) / (1024**2):.1f} MB")


## 6a. Open-loop eval -- val split (real generalization, D16) + a train sample (still fits well?)

Same `eval_openloop.py` from TIP-005, called on **val_eps** (episodes the
model never trained on -- this is the round's first real generalization
number) and on 2 train episodes (confirms the model still fits its training
data after the full run, same overfit-gate spirit as TIP-005 but now with
DINOv3 unfrozen). Un-normalization uses the checkpoint's own saved
action_min/max (TIP-003/005 pattern) -- `--stats_path`/`--stats_key` here
only affect how *input* state is normalized, matching how the model was
trained, exactly as in TIP-005.


In [ ]:
run([
    "python", "-m", "experiments.aloha.eval_openloop",
    "--checkpoint", final_ckpt,
    "--episodes", val_eps_str,
    "--data_root", "data/aloha",
    "--stats_path", STATS_PATH,
    "--stats_key", STATS_KEY,
    "--output_dir", "outputs/aloha_eval_full/val",
])


In [ ]:
sample_train_eps_str = ",".join(map(str, train_eps[:2]))
run([
    "python", "-m", "experiments.aloha.eval_openloop",
    "--checkpoint", final_ckpt,
    "--episodes", sample_train_eps_str,
    "--data_root", "data/aloha",
    "--stats_path", STATS_PATH,
    "--stats_key", STATS_KEY,
    "--output_dir", "outputs/aloha_eval_full/train_sample",
])


## 6b. NMSE summary -- gripper (D15) tracked separately vs the local smoke value (0.1014)

Local TIP-005 smoke result (RTX 4050, 8-episode overfit, frozen backbones):
`nmse_total=0.024`, all joints under gate except `left_gripper` at `0.1014`
(the one joint that was flirting with the 0.1 gate). This compares that same
gripper NMSE against the full A100 run's val (generalization) and train
(fit) numbers, to see whether unfreezing DINOv3 + 130 episodes actually
helped the weakest joint.


In [ ]:
import json

LOCAL_SMOKE_GRIPPER_NMSE = 0.1014  # TIP-005, episode 0, RTX 4050, frozen backbones, 8 episodes
GRIPPER_INDEX = 6
JOINT_NAMES = ["left_waist", "left_shoulder", "left_elbow", "left_forearm_roll",
               "left_wrist_angle", "left_wrist_rotate", "left_gripper"]

def print_nmse_summary(summary_path, label):
    with open(summary_path) as f:
        summary = json.load(f)
    overall = summary["overall"]
    print(f"=== {label} ===")
    print(f"episodes: {summary['episodes']}")
    print(f"nmse_total: {overall['nmse_total']:.4f}  joints_under_gate: {overall['joints_under_gate']}/7")
    print(f"{'joint':<18}{'NMSE':>10}")
    for j, name in enumerate(JOINT_NAMES):
        print(f"{name:<18}{overall['nmse_per_joint'][j]:>10.4f}")
    gripper_nmse = overall["nmse_per_joint"][GRIPPER_INDEX]
    delta = gripper_nmse - LOCAL_SMOKE_GRIPPER_NMSE
    direction = "IMPROVED" if delta < 0 else "WORSE" if delta > 0 else "UNCHANGED"
    print(f"\ngripper NMSE: {gripper_nmse:.4f} vs local smoke {LOCAL_SMOKE_GRIPPER_NMSE:.4f} "
          f"({direction}, delta={delta:+.4f})")
    print()
    return overall

val_overall = print_nmse_summary("outputs/aloha_eval_full/val/openloop_summary.json", "VAL (generalization)")
train_overall = print_nmse_summary("outputs/aloha_eval_full/train_sample/openloop_summary.json", "TRAIN sample (fit)")


In [ ]:
# Display the predicted-vs-GT plots inline.
from IPython.display import Image, display
import glob as _glob

for png in sorted(_glob.glob("outputs/aloha_eval_full/val/openloop_*.png")):
    print(png)
    display(Image(filename=png))
for png in sorted(_glob.glob("outputs/aloha_eval_full/train_sample/openloop_*.png")):
    print(png)
    display(Image(filename=png))


## 6c. Latency on A100 -- bf16 AND fp32 (D17: does the RTX 4050 crossover flip?)

TIP-006 found fp32 slightly *faster* than bf16 at batch=1 on a laptop RTX
4050 (~33.1ms fp32 vs ~37.0ms bf16, mean, end-to-end). A100 has much
stronger native bf16 tensor-core throughput, so this checks whether that
crossover flips on real datacenter hardware.


In [ ]:
run([
    "python", "-m", "experiments.aloha.profile_latency",
    "--per-layer", "--precision", "bf16",
    "--init_ckpt", final_ckpt,
    "--output_dir", "outputs/aloha_latency_a100",
])
run([
    "python", "-m", "experiments.aloha.profile_latency",
    "--per-layer", "--precision", "fp32",
    "--init_ckpt", final_ckpt,
    "--output_dir", "outputs/aloha_latency_a100",
])


In [ ]:
import csv

# TIP-006 reference numbers, RTX 4050 laptop, batch=1, warmup=10, iters=50 (see that Completion Report).
RTX_4050_REFERENCE = {
    "bf16": {"e2e_mean_ms": 37.014, "hz": 27.02, "vram_gb": 1.044},
    "fp32": {"e2e_mean_ms": 33.105, "hz": 30.21, "vram_gb": 1.019},
}

def read_e2e_row(csv_path):
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            if row["type"] == "component" and row["name"] == "end_to_end":
                return row
    raise ValueError(f"no end_to_end row found in {csv_path}")

print(f"{'precision':<10}{'A100 e2e_ms':>14}{'A100 Hz':>10}{'RTX4050 e2e_ms':>16}{'RTX4050 Hz':>12}{'A100 speedup':>14}")
for precision in ["bf16", "fp32"]:
    row = read_e2e_row(f"outputs/aloha_latency_a100/latency_{precision}.csv")
    a100_e2e = float(row["mean_ms"])
    a100_hz = 1000.0 / a100_e2e
    ref = RTX_4050_REFERENCE[precision]
    speedup = ref["e2e_mean_ms"] / a100_e2e
    print(f"{precision:<10}{a100_e2e:>14.3f}{a100_hz:>10.2f}{ref['e2e_mean_ms']:>16.3f}{ref['hz']:>12.2f}{speedup:>13.2f}x")

bf16_e2e = float(read_e2e_row("outputs/aloha_latency_a100/latency_bf16.csv")["mean_ms"])
fp32_e2e = float(read_e2e_row("outputs/aloha_latency_a100/latency_fp32.csv")["mean_ms"])
if bf16_e2e < fp32_e2e:
    print(f"\nOn A100: bf16 IS faster than fp32 ({bf16_e2e:.3f}ms vs {fp32_e2e:.3f}ms) -- "
          f"crossover FLIPPED vs the RTX 4050 local result.")
else:
    print(f"\nOn A100: fp32 is still faster or equal to bf16 ({fp32_e2e:.3f}ms vs {bf16_e2e:.3f}ms) -- "
          f"crossover did NOT flip vs the RTX 4050 local result.")


## 7. Package + download results to your local machine (NO Google Drive mount)

**Colab disconnecting or idle-timing-out loses everything in `/content`.
Download the zip and checkpoint NOW, right after this section runs -- don't
leave the runtime idle.**


In [ ]:
print("=" * 70)
print("REMINDER: download the outputs below RIGHT NOW.")
print("If this Colab runtime disconnects or idle-times-out before you do,")
print("the checkpoint, eval JSON/PNGs, and latency CSVs are gone for good")
print("(no Drive mount was used in this notebook).")
print("=" * 70)


In [ ]:
import shutil
from google.colab import files

ARCHIVE_BASENAME = "aloha_results"
shutil.make_archive(ARCHIVE_BASENAME, "zip", "outputs")
print(f"packaged: {ARCHIVE_BASENAME}.zip "
      f"({os.path.getsize(ARCHIVE_BASENAME + '.zip') / (1024**2):.1f} MB) "
      f"from outputs/ (eval JSON+PNG, latency CSV)")
files.download(f"{ARCHIVE_BASENAME}.zip")


### Checkpoint download

`files.download` of a large file (checkpoint with optimizer+scheduler state
is typically several hundred MB) over the browser can be slow or drop. Two
options, both shown below -- pick based on the actual file size printed:

1. **Small enough (roughly <200MB) or a stable connection:** direct
   `files.download(final_ckpt)`.
2. **Large / flaky connection:** either (a) push to a private HF Hub repo
   with `huggingface_hub.upload_file` and pull it down later from anywhere,
   or (b) save a slim copy with only `model_state_dict` (dropping
   optimizer/scheduler state, which roughly halves-to-thirds the file size
   and is all you need for eval/deploy, not for resuming training) and
   download that instead.


In [ ]:
import torch

SIZE_WARN_THRESHOLD_MB = 200
ckpt_size_mb = os.path.getsize(final_ckpt) / (1024 ** 2)
print(f"{final_ckpt}: {ckpt_size_mb:.1f} MB")

if ckpt_size_mb <= SIZE_WARN_THRESHOLD_MB:
    print("Small enough for a direct download.")
else:
    print(f"Over {SIZE_WARN_THRESHOLD_MB}MB -- consider the slim-checkpoint or HF Hub options below "
          f"instead of (or in addition to) the direct download.")


In [ ]:
# Option 1: direct download (try this first regardless of size; fall back to
# one of the options below only if it's slow/unreliable in practice).
files.download(final_ckpt)


In [ ]:
# Option 2a: slim checkpoint (model weights only, no optimizer/scheduler state).
slim_ckpt_path = final_ckpt.replace(".pth", "_slim.pth")
full_ckpt = torch.load(final_ckpt, map_location="cpu")
torch.save({
    "global_step": full_ckpt["global_step"],
    "model_state_dict": full_ckpt["model_state_dict"],
    "model_config": full_ckpt["model_config"],
    "stats": full_ckpt["stats"],
    "args": full_ckpt["args"],
}, slim_ckpt_path)
print(f"{slim_ckpt_path}: {os.path.getsize(slim_ckpt_path) / (1024**2):.1f} MB "
      f"(vs {ckpt_size_mb:.1f} MB full)")
files.download(slim_ckpt_path)


In [ ]:
# Option 2b: HF Hub fallback (uncomment and fill in a private repo you own).
# from huggingface_hub import HfApi, create_repo
# HF_CHECKPOINT_REPO = "your-username/turbovla-aloha-full"  # EDIT THIS
# create_repo(HF_CHECKPOINT_REPO, private=True, exist_ok=True)
# HfApi().upload_file(
#     path_or_fileobj=final_ckpt,
#     path_in_repo=os.path.basename(final_ckpt),
#     repo_id=HF_CHECKPOINT_REPO,
# )
# print(f"uploaded to https://huggingface.co/{HF_CHECKPOINT_REPO}")


---
Done. See the TIP-007 Completion Report for how to interpret these results
(gate pass/fail on val NMSE + gripper D15 comparison + A100 bf16/fp32
crossover) before TIP-008 VERIFY.
